In [3]:
import os
import shutil
import re
from pathlib import Path
import pandas as pd

# =========================
# Paths
# =========================
csv_path = Path("573_Gloss_List.csv")
source_1 = Path(r"D:\150 words sample\cleaned_landmarks_147words")
source_2 = Path(r"D:\ASL Citizen\data\raw_landmarks")

output_dir = Path("raw_landmarks")
output_dir.mkdir(exist_ok=True)

# =========================
# Load gloss list
# =========================
df = pd.read_csv(csv_path)

glosses = (
    df["Gloss"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.replace("_", " ")  # Ensures underscores are spaces
    .str.replace("-", " ")
    .str.strip()
    .tolist()
)

# =========================
# Helpers
# =========================
def clean_stem(stem):
    # Standardize incoming file names to match the cleaned glosses
    stem = stem.upper().replace("_", " ").replace("-", " ").strip()
    stem = re.sub(r"\s+\d+$", "", stem)
    return stem.strip()

def find_files(gloss, folder):
    matches = []
    if not folder.exists():
        return matches

    gloss = gloss.upper().strip()
    for file in folder.rglob("*.npy"):
        stem = clean_stem(file.stem)
        if stem == gloss:
            matches.append(file)
    return matches

# =========================
# Main processing
# =========================
results = []
missing = []

for gloss in glosses:
    gloss = gloss.upper().strip()
    files = find_files(gloss, source_1)

    if not files:
        files = find_files(gloss, source_2)

    if not files:
        missing.append(gloss)
        continue

    copied_names = []

    for i, f in enumerate(files):
        # Use a space and index to avoid underscores entirely
        # Using f"{gloss} {i}.npy" ensures every file has a unique name 
        # without triggering your underscore detection script.
        new_name = f"{gloss} {i}.npy"
        dest = output_dir / new_name

        shutil.copy2(f, dest)
        copied_names.append(dest.name)

    results.append({
        "gloss": gloss,
        "count": len(files),
        "files_names": ";".join(copied_names)
    })

# =========================
# Save outputs
# =========================
final_df = pd.DataFrame(results)
final_df.to_csv("final_gloss_files.csv", index=False)

missing_df = pd.DataFrame({"missing_gloss": missing})
missing_df.to_csv("missing_glosses.csv", index=False)

print("Done!")
print(f"Copied glosses: {len(final_df)}")
print(f"Missing glosses: {len(missing)}")

Done!
Copied glosses: 501
Missing glosses: 72
